# 18 — Debug a solver from evidence

**Prerequisites:** notebooks 09–11: NLP dictionaries, bounds, and KKT conditions.
**Outcome:** distinguish a model evaluation error, an infeasible formulation, and a
convergence failure; inspect physical residuals before accepting a result.

A solver returns a candidate point and a status. Neither a plausible plot nor a
nonempty `x` array establishes feasibility. Work in this order:

1. Check dimensions, units, parameter values, bounds, and initial evaluations.
2. Inspect individual residuals and the solver's iteration history.
3. Check KKT conditions, scaling, derivatives, and initialization.
4. Change one option only after identifying evidence that motivates it.

The core path below needs only basic calculus. **Deeper checks** explain multipliers
and source mapping. Notebook **18a** contains controlled failure-and-repair labs
for domains, scaling, derivatives, rank, nonsmoothness, and iteration limits.


## 1. Set up independent diagnostic and plotting tools

This cell locates the repository and imports only presentation/diagnostic helpers.
It does not import an optimization model. `perf_counter` measures the whole solve;
plugins do not consistently report aggregate wall time. Plotting supports hover
and zoom when Plotly is installed; `CASADI_TUTORIAL_PLOTS=static` selects Matplotlib.


In [ ]:
# Locate the checkout without installing a model package.
import sys
from pathlib import Path

repository_root = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'casadi_tutorial').is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

import casadi as ca
import numpy as np
from time import perf_counter

from casadi_tutorial.diagnostics import bound_report, kkt_residuals, solver_summary
from casadi_tutorial.interactive import experiment
from casadi_tutorial.plotting import display_plot, plot_series


## 2. Write the problem and its row ordering

For decisions $x=(x_0,x_1)$ and fixed parameter $b$, solve

$$
\min_x (1-x_0)^2+100(x_1-x_0^2)^2,
\quad x_0+x_1=b,\quad -2\le x_i\le2.
$$

The equality has **one row**, so its bounds and multiplier have one entry. The
parameter changes the problem's numeric data, not the symbolic graph. `evaluate`
returns objective, constraints, objective gradient, and constraint Jacobian;
checking all four matters because a finite function may have an invalid derivative.


In [ ]:
x = ca.MX.sym('x', 2)
b = ca.MX.sym('b')
f = (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2
g = x[0] + x[1]
problem = {'x': x, 'p': b, 'f': f, 'g': g}
evaluate = ca.Function('evaluate_rosenbrock', [x, b],
                       [f, g, ca.gradient(f, x), ca.jacobian(g, x)])
constraint_labels = ['sum of decisions']
lower_x, upper_x = np.full(2, -2.0), np.full(2, 2.0)
initial = np.array([0.0, 1.0])


## 3. Inspect the initial point before asking IPOPT to move

Flattening uses CasADi's column-major ordering. A bound report preserves each
row's index, name, value, and violation; invalid numeric values have infinite
violation. An infeasible initial guess is allowed by IPOPT, but you should know
which constraints it violates. Here the initial guess satisfies the equality.


In [ ]:
initial_values = evaluate(initial, 1.0)
for name, value in zip(['objective', 'constraints', 'gradient', 'Jacobian'], initial_values):
    array = np.asarray(value.full())
    print(name, 'shape:', array.shape, 'finite:', np.isfinite(array).all())
    assert np.isfinite(array).all()
print('decision bounds:', bound_report(initial, lower_x, upper_x,
                                      labels=['x[0]', 'x[1]']))
print('constraint bounds:', bound_report(initial_values[1], 1, 1,
                                        labels=constraint_labels))


## 4. Solve, retain the evidence, and calculate physical residuals

`error_on_fail=False` lets this teaching function return a failed candidate for
inspection. It does **not** accept that candidate for application use. The caller
must check `summary['success']` and independent residuals. Ordinary applications
can keep exceptions enabled and collect the same evidence in `except`.

For CasADi's net signed multipliers, stationarity is

$$\nabla f+J_g^\mathsf{T}\lambda_g+\lambda_x=0.$$

A negative bound multiplier acts on a lower bound; a positive one acts on an upper
bound. The four optional decision-bound inputs are supplied together so active
variable bounds are included rather than silently omitted. All residual norms
below use the original formulation, not IPOPT's internal scaling.


In [ ]:
def solve_case(target=1.0, initial_x0=0.0, max_iterations=100):
    start = np.array([initial_x0, 1.0])
    solver = ca.nlpsol('debug_rosenbrock', 'ipopt', problem, {
        'print_time': False, 'error_on_fail': False,
        'ipopt.print_level': 0, 'ipopt.max_iter': int(max_iterations),
        'ipopt.tol': 1e-9,
    })
    before = perf_counter()
    result = solver(x0=start, p=target, lbx=lower_x, ubx=upper_x,
                    lbg=target, ubg=target)
    elapsed = perf_counter() - before
    stats = solver.stats()
    candidate = np.asarray(result['x'].full()).reshape(-1)
    _, constraint_value, gradient, jacobian = evaluate(candidate, target)
    residuals = kkt_residuals(
        gradient, jacobian, result['lam_g'], constraint_value, [target], [target],
        variable=candidate, lower_x=lower_x, upper_x=upper_x,
        lambda_x=result['lam_x'],
    )
    rows = bound_report(constraint_value, target, target, labels=constraint_labels)
    rows += bound_report(candidate, lower_x, upper_x, labels=['x[0]', 'x[1]'])
    return {'candidate': candidate, 'objective': float(result['f']), 'stats': stats,
            'summary': solver_summary(stats, elapsed_s=elapsed), 'kkt': residuals,
            'rows': rows, 'lam_g': result['lam_g'], 'lam_x': result['lam_x']}

baseline = solve_case()
print('candidate:', baseline['candidate'], 'objective:', baseline['objective'])
print('status:', baseline['summary'])
print('KKT:', baseline['kkt'])
assert baseline['summary']['success']
assert max(baseline['kkt'].values()) < 1e-6


## 5. Read the convergence trace

The trace is post-solve data, so drawing it cannot slow the solver. IPOPT reports
primal and dual infeasibility on its own internal definitions/scales; compare
the final point using the independently calculated KKT report above. The plotting
floor $10^{-16}$ only makes zero visible on a log axis; it does not change data used
for acceptance. Objective values need not decrease on every constrained step.


In [ ]:
def show_evidence(report):
    print(report['summary'])
    print('KKT:', report['kkt'])
    for row in sorted(report['rows'], key=lambda row: row['violation'], reverse=True):
        print(row)
    history = report['stats'].get('iterations', {})
    curves = {key: np.maximum(np.asarray(history[key]), 1e-16)
              for key in ('inf_pr', 'inf_du') if key in history}
    if curves:
        display_plot(plot_series(np.arange(len(next(iter(curves.values())))), curves,
                                 title='IPOPT iteration residuals', xlabel='iteration',
                                 ylabel='reported infeasibility', log_y=True))
    else:
        print('Iteration history was not reported by this solver plugin.')

show_evidence(baseline)


## 6. Prove infeasibility before tuning the solver

With $b=5$, the equality asks for $x_0+x_1=5$, while the two upper bounds imply
$x_0+x_1\le4$. This contradiction proves the feasible set is empty. A longer
iteration budget cannot repair it. The experiment keeps the failed candidate and
its residuals; it does not assert an exact status string, since equivalent failures
can be reported differently across solver builds.


In [ ]:
infeasible = solve_case(target=5.0, max_iterations=40)
show_evidence(infeasible)
assert not infeasible['summary']['success']
assert infeasible['kkt']['primal_inf'] > 0.1


## 7. Deeper check: map flattened rows back to source with Opti

The following independent scalar example deliberately adds `x >= 3` and `x <= 1`.
`g_describe` connects a row index from a warning to the line that created it.
`debug.value(..., initial())` is available before a solve; `show_infeasibilities`
uses the last iterate after a failed solve. Keep the original traceback in a real
bug report: the compact print here is only to keep a known-failure lesson readable.


In [ ]:
optimization = ca.Opti()
scalar = optimization.variable()
optimization.minimize((scalar - 2)**2)
optimization.subject_to(scalar >= 3)
optimization.subject_to(scalar <= 1)
optimization.set_initial(scalar, 2)
optimization.solver('ipopt', {'print_time': False}, {'print_level': 0, 'max_iter': 20})
print('initial constraints:', optimization.debug.value(optimization.g, optimization.initial()))
print('row 0:', optimization.debug.g_describe(0))
try:
    optimization.solve()
except RuntimeError as error:
    print('Expected failure:', str(error).splitlines()[-1])
    print(solver_summary(optimization.stats()))
    optimization.debug.show_infeasibilities(1e-8)
else:
    raise AssertionError('Contradictory constraints unexpectedly accepted.')


## 8. Interactive diagnosis

Predict the outcome before running: try target `5`, then restore `1`; or keep a
feasible target and set the iteration limit to `1`. Change the first initial
decision to explore convergence from another starting point. Controls never run a
solve automatically. **Reset controls** restores values and waits for **Run**.
The static site retains the computed default plots; new solves require JupyterLab.


In [ ]:
def run_diagnosis(target, initial_x0, max_iterations):
    show_evidence(solve_case(target, initial_x0, max_iterations))

experiment(run_diagnosis, {'target': 1.0, 'initial_x0': 0.0, 'max_iterations': 100},
           title='Change the formulation, guess, or iteration budget')


## Recap and checkpoint

| Evidence | Next useful check |
|---|---|
| Invalid initial function/derivative | Isolate the expression and its real domain |
| Contradictory bounds or balance requirements | Repair the formulation or operating request |
| Large physical primal residual | Inspect named rows and initial-state consistency |
| Tiny primal residual but large stationarity | Check bound multipliers, scaling, derivatives |
| Iteration limit with declining residuals | Improve initialization; then assess the budget |
| Restoration or singular-system status | Inspect domains, redundant rows, curvature and scaling |

A return status suggests an investigation; it does not prove the root cause.
Complementarity here uses split net signed multipliers, distinct from IPOPT's
internal barrier statistics. KKT conditions are local first-order conditions;
nonconvexity and constraint qualifications still matter.

**Checkpoint:** explain why `success=True` alone is insufficient, why a failed
candidate is useful, and why `lam_x` must appear when variable bounds are active.
Then complete Exercise 04 and the deeper failure labs in 18a.

References: [CasADi debugging](https://web.casadi.org/docs/#extras),
[IPOPT output](https://coin-or.github.io/Ipopt/OUTPUT.html),
[IPOPT diagnostic options](https://coin-or.github.io/Ipopt/OPTIONS.html).
